In [ ]:

"""
CN7 11 - Test Mapping 확정 + Train/Test Shift + Exact Overlap
==============================================================

목적
----
1) 복원된 test metadata가 row-level로도 신뢰 가능한지 확인
   - raw S14 subset 원래 순서와 final test 순서가 동일한지
   - Unnamed/index 계열 컬럼이 source row를 보존하는지

2) train raw 1211 vs test raw 35239 분포 차이 진단
   - KS
   - normalized Wasserstein
   - standardized mean difference
   - PSI
   - train range 밖 test 비율

3) train/test exact sensor vector overlap 확인
   - test 중 train과 완전히 같은 sensor vector 비율
   - train positive vector와 겹치는 test 개수
   - train negative vector와 겹치는 test 개수
   - train에서 same-X/different-target인 vector와 test overlap

중요
----
- 모델 학습 없음
- 빠른 구조 진단용
"""

from pathlib import Path
from collections import Counter
import warnings
import numpy as np
import pandas as pd

from scipy.stats import ks_2samp, wasserstein_distance

warnings.filterwarnings("ignore")


# =============================================================================
# 0. 경로
# =============================================================================

BASE_DIR = Path(
    r"../../../data/raw"
)

CN7_WORK_DIR = Path(
    r"../../outputs/research"
)

TRAIN_PATH = (
    CN7_WORK_DIR
    / "cn7_03_eda"
    / "cn7_train_raw_1211.csv"
)

TEST_RESTORED_PATH = (
    CN7_WORK_DIR
    / "cn7_10_test_restore"
    / "cn7_test_restored_35239.csv"
)

RAW_UNLABELED_PATH = (
    BASE_DIR
    / "unlabeled_data.csv"
)

FINAL_TEST_PATH = (
    BASE_DIR
    / "moldset_unlabeled_cn7.csv"
)

OUT_DIR = (
    CN7_WORK_DIR
    / "cn7_11_shift_overlap"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TARGET = "PassOrFail"

SENSOR_COLS = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Clamp_Open_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4",
]


# =============================================================================
# 유틸
# =============================================================================

def banner(title):
    print("\n" + "=" * 118)
    print(title)
    print("=" * 118)


def standardize_raw(df, cols):
    x = df[cols].astype(float).to_numpy()

    mean = np.nanmean(
        x,
        axis=0,
    )

    std = np.nanstd(
        x,
        axis=0,
        ddof=0,
    )

    z = np.full_like(
        x,
        np.nan,
        dtype=float,
    )

    nonzero = (
        std > 1e-15
    )

    z[:, nonzero] = (
        x[:, nonzero]
        - mean[nonzero]
    ) / std[nonzero]

    z[:, ~nonzero] = 0.0

    return z, mean, std


def psi_numeric(train, test, bins=10):
    """
    Train quantile 기준 PSI.
    extreme 값 포함을 위해 -inf/+inf 사용.
    """
    train = np.asarray(train, dtype=float)
    test = np.asarray(test, dtype=float)

    train = train[
        np.isfinite(train)
    ]

    test = test[
        np.isfinite(test)
    ]

    if (
        len(train) == 0
        or len(test) == 0
    ):
        return np.nan

    quantiles = np.linspace(
        0,
        1,
        bins + 1,
    )

    edges = np.unique(
        np.quantile(
            train,
            quantiles,
        )
    )

    if len(edges) < 3:
        return 0.0

    edges[0] = -np.inf
    edges[-1] = np.inf

    train_hist, _ = np.histogram(
        train,
        bins=edges,
    )

    test_hist, _ = np.histogram(
        test,
        bins=edges,
    )

    train_rate = (
        train_hist
        / train_hist.sum()
    )

    test_rate = (
        test_hist
        / test_hist.sum()
    )

    eps = 1e-6

    train_rate = np.clip(
        train_rate,
        eps,
        None,
    )

    test_rate = np.clip(
        test_rate,
        eps,
        None,
    )

    return float(
        np.sum(
            (
                test_rate
                - train_rate
            )
            * np.log(
                test_rate
                / train_rate
            )
        )
    )


def rounded_keys(df, cols, decimals=8):
    arr = (
        df[cols]
        .astype(float)
        .to_numpy()
    )

    arr = np.round(
        arr,
        decimals,
    )

    return [
        tuple(row)
        for row in arr
    ]


# =============================================================================
# 1. 데이터 로드
# =============================================================================

banner(
    "CN7 11 - Test Mapping / Shift / Overlap"
)

train = pd.read_csv(
    TRAIN_PATH
)

test_restored = pd.read_csv(
    TEST_RESTORED_PATH
)

raw_unlabeled = pd.read_csv(
    RAW_UNLABELED_PATH
)

final_test = pd.read_csv(
    FINAL_TEST_PATH
)

print(
    "train:",
    train.shape,
)

print(
    "test restored:",
    test_restored.shape,
)

print(
    "raw unlabeled:",
    raw_unlabeled.shape,
)

print(
    "final test:",
    final_test.shape,
)


# =============================================================================
# 2. raw S14 subset 직접 재구성
# =============================================================================

banner(
    "1) S14 raw subset 재구성"
)

if "EQUIP_CD" not in raw_unlabeled.columns:
    raise KeyError(
        "raw unlabeled에 EQUIP_CD 없음"
    )

cn7_mask = (
    raw_unlabeled["PART_NAME"]
    .astype(str)
    .str.contains(
        "CN7",
        case=False,
        na=False,
    )
)

s14_mask = (
    raw_unlabeled["EQUIP_CD"]
    .astype(str)
    .eq("S14")
)

source_s14 = (
    raw_unlabeled.loc[
        cn7_mask & s14_mask
    ]
    .copy()
)

source_s14[
    "__source_row_index__"
] = source_s14.index

source_s14 = (
    source_s14
    .reset_index(
        drop=True
    )
)

print(
    "CN7 rows:",
    int(cn7_mask.sum()),
)

print(
    "CN7 + S14 rows:",
    len(
        source_s14
    ),
)

print(
    "final test rows:",
    len(
        final_test
    ),
)

if len(
    source_s14
) != len(
    final_test
):
    raise RuntimeError(
        "S14 행 수와 final test 행 수 불일치"
    )


# =============================================================================
# 3. source order vs final test order 확인
# =============================================================================

banner(
    "2) Final test row order 보존 여부"
)

sensor_cols = [
    c
    for c in SENSOR_COLS
    if (
        c in source_s14.columns
        and c in final_test.columns
    )
]

z, mean, std = standardize_raw(
    source_s14,
    sensor_cols,
)

target_matrix = (
    final_test[
        sensor_cols
    ]
    .astype(float)
    .to_numpy()
)

diff = np.abs(
    z - target_matrix
)

rowwise_mae = np.nanmean(
    diff,
    axis=1,
)

rowwise_max = np.nanmax(
    diff,
    axis=1,
)

same_order_rate_10 = float(
    (
        rowwise_max
        < 1e-10
    ).mean()
)

same_order_rate_8 = float(
    (
        rowwise_max
        < 1e-8
    ).mean()
)

print(
    "rowwise MAE median:",
    float(
        np.nanmedian(
            rowwise_mae
        )
    ),
)

print(
    "rowwise MAX:",
    float(
        np.nanmax(
            rowwise_max
        )
    ),
)

print(
    "same-order exact rate (<1e-10):",
    same_order_rate_10,
)

print(
    "same-order exact rate (<1e-8):",
    same_order_rate_8,
)

order_preserved = (
    same_order_rate_10
    > 0.999999
)

print(
    "order_preserved:",
    order_preserved,
)


# =============================================================================
# 4. index-like 컬럼 확인
# =============================================================================

banner(
    "3) Index-like 컬럼 진단"
)

index_like_cols = [
    c
    for c in final_test.columns
    if (
        "unnamed" in c.lower()
        or c.lower() in [
            "index",
            "idx",
            "id",
        ]
    )
]

print(
    "final test index-like:",
    index_like_cols,
)

for c in index_like_cols:

    print()

    print(
        f"[{c}]"
    )

    print(
        final_test[c]
        .head(10)
        .to_string(
            index=False
        )
    )

    vals = pd.to_numeric(
        final_test[c],
        errors="coerce",
    )

    print(
        "numeric rate:",
        vals.notna().mean(),
    )

    if vals.notna().all():

        print(
            "min/max:",
            vals.min(),
            vals.max(),
        )

        print(
            "unique:",
            vals.nunique(),
        )


# =============================================================================
# 5. train / test metadata 범위
# =============================================================================

banner(
    "4) Train / Test metadata"
)

for name, d in [
    (
        "TRAIN",
        train,
    ),
    (
        "TEST",
        source_s14,
    ),
]:

    print(
        f"\n[{name}]"
    )

    if "TimeStamp" in d.columns:

        ts = pd.to_datetime(
            d[
                "TimeStamp"
            ],
            errors="coerce",
        )

        print(
            "TimeStamp:",
            ts.min(),
            "~",
            ts.max(),
        )

    if (
        "PART_FACT_PLAN_DATE"
        in d.columns
    ):

        plan = pd.to_datetime(
            d[
                "PART_FACT_PLAN_DATE"
            ],
            errors="coerce",
        )

        print(
            "PlanDate:",
            plan.min(),
            "~",
            plan.max(),
        )

        print(
            "unique plan dates:",
            plan.dt.date.nunique(),
        )

    if "PART_FACT_SERIAL" in d.columns:

        print(
            "serial unique:",
            d[
                "PART_FACT_SERIAL"
            ].nunique(
                dropna=False
            ),
        )


# =============================================================================
# 6. Side 분포
# =============================================================================

banner(
    "5) Side 분포"
)

def infer_side(d):
    if "Side" in d.columns:
        return d["Side"].astype(str)

    if "PART_NAME" in d.columns:
        s = (
            d["PART_NAME"]
            .astype(str)
            .str.upper()
        )

        return pd.Series(
            np.where(
                s.str.contains(
                    r"\bRH\b",
                    regex=True,
                    na=False,
                ),
                "RH",
                np.where(
                    s.str.contains(
                        r"\bLH\b",
                        regex=True,
                        na=False,
                    ),
                    "LH",
                    "UNKNOWN",
                ),
            ),
            index=d.index,
        )

    return pd.Series(
        "UNKNOWN",
        index=d.index,
    )


train_side = infer_side(
    train
)

test_side = infer_side(
    source_s14
)

print(
    "TRAIN:"
)

print(
    train_side
    .value_counts()
    .to_string()
)

print()

print(
    "TEST:"
)

print(
    test_side
    .value_counts()
    .to_string()
)


# =============================================================================
# 7. Sensor shift
# =============================================================================

banner(
    "6) Train/Test sensor shift"
)

shift_rows = []

for c in sensor_cols:

    tr = pd.to_numeric(
        train[c],
        errors="coerce",
    ).dropna()

    te = pd.to_numeric(
        source_s14[c],
        errors="coerce",
    ).dropna()

    tr_mean = float(
        tr.mean()
    )

    te_mean = float(
        te.mean()
    )

    tr_std = float(
        tr.std(
            ddof=0
        )
    )

    smd = (
        (
            te_mean
            - tr_mean
        )
        / tr_std
        if tr_std > 1e-12
        else np.nan
    )

    try:
        ks = ks_2samp(
            tr,
            te,
        )

        ks_stat = float(
            ks.statistic
        )

        ks_p = float(
            ks.pvalue
        )

    except Exception:

        ks_stat = np.nan
        ks_p = np.nan

    w = float(
        wasserstein_distance(
            tr,
            te,
        )
    )

    norm_w = (
        w / tr_std
        if tr_std > 1e-12
        else np.nan
    )

    train_min = float(
        tr.min()
    )

    train_max = float(
        tr.max()
    )

    outside_rate = float(
        (
            (te < train_min)
            |
            (te > train_max)
        ).mean()
    )

    shift_rows.append({
        "feature":
            c,
        "train_mean":
            tr_mean,
        "test_mean":
            te_mean,
        "train_std":
            tr_std,
        "SMD_test_minus_train":
            smd,
        "KS_stat":
            ks_stat,
        "KS_pvalue":
            ks_p,
        "Wasserstein":
            w,
        "Norm_Wasserstein":
            norm_w,
        "PSI":
            psi_numeric(
                tr,
                te,
                bins=10,
            ),
        "Test_Outside_Train_Range_Rate":
            outside_rate,
    })

shift_df = (
    pd.DataFrame(
        shift_rows
    )
)

shift_df[
    "abs_SMD"
] = shift_df[
    "SMD_test_minus_train"
].abs()

shift_df = (
    shift_df
    .sort_values(
        [
            "KS_stat",
            "abs_SMD",
        ],
        ascending=False,
    )
)

print(
    shift_df[
        [
            "feature",
            "SMD_test_minus_train",
            "KS_stat",
            "Norm_Wasserstein",
            "PSI",
            "Test_Outside_Train_Range_Rate",
        ]
    ]
    .round(4)
    .to_string(
        index=False
    )
)

shift_df.to_csv(
    OUT_DIR
    / "train_test_sensor_shift.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 8. Exact sensor overlap
# =============================================================================

banner(
    "7) Train/Test exact sensor-vector overlap"
)

usable_sensor_cols = [
    c
    for c in sensor_cols
    if (
        train[c].nunique(
            dropna=False
        ) > 1
    )
]

train_keys = rounded_keys(
    train,
    usable_sensor_cols,
    decimals=8,
)

test_keys = rounded_keys(
    source_s14,
    usable_sensor_cols,
    decimals=8,
)

train_counter = Counter(
    train_keys
)

test_counter = Counter(
    test_keys
)

train_unique = set(
    train_counter
)

test_unique = set(
    test_counter
)

common_unique = (
    train_unique
    & test_unique
)

test_overlap_rows = sum(
    count
    for key, count
    in test_counter.items()
    if key in train_unique
)

train_overlap_rows = sum(
    count
    for key, count
    in train_counter.items()
    if key in test_unique
)

print(
    "train unique vectors:",
    len(
        train_unique
    ),
)

print(
    "test unique vectors:",
    len(
        test_unique
    ),
)

print(
    "common unique vectors:",
    len(
        common_unique
    ),
)

print(
    "test rows exact-overlap with train:",
    test_overlap_rows,
    "/",
    len(source_s14),
    "=",
    test_overlap_rows
    / len(source_s14),
)

print(
    "train rows exact-overlap with test:",
    train_overlap_rows,
    "/",
    len(train),
    "=",
    train_overlap_rows
    / len(train),
)


# =============================================================================
# 9. Train target별 vector overlap
# =============================================================================

banner(
    "8) Positive/Negative vector overlap"
)

train_tmp = train.copy()

train_tmp[
    "__key__"
] = train_keys

vector_target = (
    train_tmp
    .groupby(
        "__key__"
    )[TARGET]
    .agg(
        [
            "count",
            "sum",
            "nunique",
        ]
    )
)

positive_only_keys = set(
    vector_target.index[
        (
            vector_target[
                "sum"
            ] > 0
        )
        &
        (
            vector_target[
                "nunique"
            ] == 1
        )
    ]
)

negative_only_keys = set(
    vector_target.index[
        (
            vector_target[
                "sum"
            ] == 0
        )
        &
        (
            vector_target[
                "nunique"
            ] == 1
        )
    ]
)

conflict_keys = set(
    vector_target.index[
        vector_target[
            "nunique"
        ] > 1
    ]
)

def count_test_rows_for_keyset(
    keyset
):
    return int(
        sum(
            test_counter[
                key
            ]
            for key in keyset
            if key in test_counter
        )
    )


positive_overlap = (
    count_test_rows_for_keyset(
        positive_only_keys
    )
)

negative_overlap = (
    count_test_rows_for_keyset(
        negative_only_keys
    )
)

conflict_overlap = (
    count_test_rows_for_keyset(
        conflict_keys
    )
)

print(
    "positive-only train vectors:",
    len(
        positive_only_keys
    ),
)

print(
    "negative-only train vectors:",
    len(
        negative_only_keys
    ),
)

print(
    "conflict train vectors:",
    len(
        conflict_keys
    ),
)

print()

print(
    "test rows matching positive-only vector:",
    positive_overlap,
)

print(
    "test rows matching negative-only vector:",
    negative_overlap,
)

print(
    "test rows matching conflict vector:",
    conflict_overlap,
)

overlap_summary = pd.DataFrame([
    {
        "category":
            "positive_only",
        "train_unique_vectors":
            len(
                positive_only_keys
            ),
        "test_overlap_rows":
            positive_overlap,
    },
    {
        "category":
            "negative_only",
        "train_unique_vectors":
            len(
                negative_only_keys
            ),
        "test_overlap_rows":
            negative_overlap,
    },
    {
        "category":
            "same_X_different_target",
        "train_unique_vectors":
            len(
                conflict_keys
            ),
        "test_overlap_rows":
            conflict_overlap,
    },
])

overlap_summary.to_csv(
    OUT_DIR
    / "exact_overlap_by_train_target.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 10. exact-overlap test rows 저장
# =============================================================================

banner(
    "9) Overlap test rows 저장"
)

test_overlap_df = (
    source_s14.copy()
)

test_overlap_df[
    "__key__"
] = test_keys

def key_category(key):
    if key in positive_only_keys:
        return "positive_only"

    if key in negative_only_keys:
        return "negative_only"

    if key in conflict_keys:
        return "same_X_different_target"

    return "no_exact_train_match"


test_overlap_df[
    "train_exact_match_category"
] = [
    key_category(
        key
    )
    for key in test_keys
]

print(
    test_overlap_df[
        "train_exact_match_category"
    ]
    .value_counts()
    .to_string()
)

test_overlap_df = (
    test_overlap_df
    .drop(
        columns=[
            "__key__"
        ]
    )
)

test_overlap_df.to_csv(
    OUT_DIR
    / "test_rows_with_overlap_category.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 11. 자동 요약
# =============================================================================

banner(
    "10) 자동 요약"
)

high_shift = shift_df[
    (
        shift_df[
            "KS_stat"
        ] >= 0.30
    )
    |
    (
        shift_df[
            "abs_SMD"
        ] >= 0.50
    )
    |
    (
        shift_df[
            "PSI"
        ] >= 0.25
    )
]

lines = []

lines.append(
    f"- Test raw subset row-order preserved: "
    f"{order_preserved}"
)

lines.append(
    f"- same-order exact rate(<1e-10): "
    f"{same_order_rate_10:.6f}"
)

lines.append(
    f"- Train rows={len(train)}, "
    f"Test rows={len(source_s14)}"
)

lines.append(
    f"- Shift 강한 센서 수: "
    f"{len(high_shift)} / {len(shift_df)}"
)

lines.append(
    f"- Test exact-overlap rows: "
    f"{test_overlap_rows} / {len(source_s14)} "
    f"({test_overlap_rows / len(source_s14):.3%})"
)

lines.append(
    f"- Test rows matching train positive-only vectors: "
    f"{positive_overlap}"
)

lines.append(
    f"- Test rows matching train negative-only vectors: "
    f"{negative_overlap}"
)

lines.append(
    f"- Test rows matching train same-X/different-target vectors: "
    f"{conflict_overlap}"
)

if order_preserved:
    lines.append(
        "- Final test가 S14 raw subset의 원래 row 순서를 보존하므로 "
        "Side/TimeStamp metadata를 row-level로 직접 사용할 수 있다."
    )
else:
    lines.append(
        "- Final test가 raw subset 순서를 그대로 보존하지 않는다. "
        "동일 sensor vector 중복이 있는 경우 row-level Side/TimeStamp 대응은 "
        "추가 검증 전까지 확정하지 않는다."
    )

for line in lines:
    print(
        line
    )

with open(
    OUT_DIR
    / "shift_overlap_summary.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write(
        "\n".join(
            lines
        )
    )


# =============================================================================
# 완료
# =============================================================================

banner(
    "완료"
)

print(
    "결과 폴더:",
    OUT_DIR,
)

print()

print(
    "핵심 파일:"
)

print(
    "  1) train_test_sensor_shift.csv"
)

print(
    "  2) exact_overlap_by_train_target.csv"
)

print(
    "  3) test_rows_with_overlap_category.csv"
)

print(
    "  4) shift_overlap_summary.txt"
)

print()

print(
    "실행 후 2) Final test row order 보존 여부, "
    "6) Train/Test sensor shift, "
    "7) exact overlap, "
    "8) Positive/Negative overlap, "
    "10) 자동 요약을 보내주세요."
)
